# 11. MCP (Model Context Protocol)

## 학습 목표

MCP(Model Context Protocol)를 통해 외부 도구와 컨텍스트를 에이전트에 연결하는 방법을 알아봅니다.

이 노트북에서 다루는 내용:
- MCP의 개념과 아키텍처(서버/클라이언트/호스트)를 이해한다
- `langchain-mcp-adapters` 패키지로 MCP 서버에 연결한다
- `ChatOpenAI.bind_tools(mcp_tools)`로 에이전트와 MCP 도구를 통합한다
- Stdio와 SSE 전송 방식의 차이를 안다
- 다중 MCP 서버를 연결하는 방법을 익힌다

## 11.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — https://lf.ddok.ai


## 11.2 MCP 개념

**MCP(Model Context Protocol)**는 외부 도구와 컨텍스트를 **표준화된 방식**으로 LLM에 제공하기 위한 오픈 프로토콜입니다.

### 아키텍처 구성 요소

| 구성 요소 | 역할 | 예시 |
|----------|------|------|
| **MCP 서버** | 도구, 리소스, 프롬프트를 노출 | 파일 시스템 서버, DB 서버, API 래퍼 |
| **MCP 클라이언트** | 서버에 연결하여 도구를 가져옴 | `MultiServerMCPClient` |
| **호스트** | 클라이언트를 관리하고 LLM과 연결 | LangChain 에이전트, IDE |

### 핵심 리소스 타입

- **Tools**: 에이전트가 호출할 수 있는 실행 가능한 함수
- **Resources**: 파일, DB 레코드 등의 데이터 (LangChain Blob 객체로 변환)
- **Prompts**: 재사용 가능한 프롬프트 템플릿

### 왜 MCP인가?

MCP 이전에는 각 도구마다 개별적으로 연결 코드를 작성해야 했습니다. MCP는 이를 **하나의 표준 프로토콜**로 통합하여:
- 도구 제공자는 한 번만 MCP 서버를 구현하면 됩니다
- LLM 호스트는 MCP 클라이언트 하나로 모든 도구에 접근할 수 있습니다
- 생태계 전체에서 도구를 재사용할 수 있습니다

## 11.3 langchain-mcp-adapters 설치

LangChain에서 MCP를 사용하려면 `langchain-mcp-adapters` 패키지가 필요합니다.

In [3]:
# MCP 어댑터 설치 명령어
print("MCP 어댑터 설치:")
print("  pip install langchain-mcp-adapters")
print()
print("주요 컴포넌트:")
print("  - MultiServerMCPClient: 여러 MCP 서버를 관리하는 클라이언트")
print("  - client.get_tools(): MCP 서버에서 도구를 LangChain Tool로 변환")
print("  - client.get_resources(): MCP 서버에서 리소스를 LangChain Blob으로 변환")

MCP 어댑터 설치:
  pip install langchain-mcp-adapters

주요 컴포넌트:
  - MultiServerMCPClient: 여러 MCP 서버를 관리하는 클라이언트
  - client.get_tools(): MCP 서버에서 도구를 LangChain Tool로 변환
  - client.get_resources(): MCP 서버에서 리소스를 LangChain Blob으로 변환


## 11.4 Stdio 전송

**Stdio(Standard I/O)** 전송은 로컬 서브프로세스를 통해 MCP 서버와 통신합니다. 개발 및 테스트 환경에 적합합니다.

In [4]:
# Stdio 전송 설정 예시 (실제 실행하지 않음)
stdio_config = {
    "math_server": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    }
}

print("Stdio 전송 설정:")
import json
print(json.dumps(stdio_config, indent=2))
print()
print("사용 패턴:")
print("  async with MultiServerMCPClient(stdio_config) as client:")
print("      tools = client.get_tools()")
print()
print("특징:")
print("  - 로컬 서브프로세스로 통신 (stdin/stdout)")
print("  - 개발/테스트 환경에 적합")
print("  - 별도 서버 실행 불필요 — 자동으로 프로세스 시작")

Stdio 전송 설정:
{
  "math_server": {
    "transport": "stdio",
    "command": "python",
    "args": [
      "math_server.py"
    ]
  }
}

사용 패턴:
  async with MultiServerMCPClient(stdio_config) as client:
      tools = client.get_tools()

특징:
  - 로컬 서브프로세스로 통신 (stdin/stdout)
  - 개발/테스트 환경에 적합
  - 별도 서버 실행 불필요 — 자동으로 프로세스 시작


## 11.5 SSE/HTTP 전송

**HTTP(streamable-http)** 전송은 웹 기반 통신으로, 원격 MCP 서버에 연결할 때 사용합니다. 인증 헤더와 커스텀 설정을 지원합니다.

In [5]:
# HTTP/streamable-http 전송 설정 예시 (실제 실행하지 않음)
http_config = {
    "weather_server": {
        "transport": "streamable_http",
        "url": "https://weather-mcp.example.com/mcp",
        "headers": {
            "Authorization": "Bearer YOUR_API_KEY"
        },
    }
}

print("HTTP 전송 설정:")
import json
print(json.dumps(http_config, indent=2))
print()
print("사용 패턴:")
print("  async with MultiServerMCPClient(http_config) as client:")
print("      tools = client.get_tools()")
print()
print("전송 방식 비교:")
print("  | 전송 방식        | 사용 사례         | 인증 지원 |")
print("  |-----------------|------------------|----------|")
print("  | stdio           | 로컬 개발/테스트   | N/A      |")
print("  | streamable_http | 원격 서버 연결     | 헤더 지원  |")
print("  | sse             | 레거시 SSE 서버   | 헤더 지원  |")

HTTP 전송 설정:
{
  "weather_server": {
    "transport": "streamable_http",
    "url": "https://weather-mcp.example.com/mcp",
    "headers": {
      "Authorization": "Bearer YOUR_API_KEY"
    }
  }
}

사용 패턴:
  async with MultiServerMCPClient(http_config) as client:
      tools = client.get_tools()

전송 방식 비교:
  | 전송 방식        | 사용 사례         | 인증 지원 |
  |-----------------|------------------|----------|
  | stdio           | 로컬 개발/테스트   | N/A      |
  | streamable_http | 원격 서버 연결     | 헤더 지원  |
  | sse             | 레거시 SSE 서버   | 헤더 지원  |


## 11.6 MCP 도구 로드 및 에이전트 통합

MCP 서버에서 가져온 도구를 LangChain 에이전트에 바인딩하는 패턴입니다.

In [6]:
# MCP 도구를 에이전트에 통합하는 패턴 (개념 코드)
print("MCP 도구 → 에이전트 통합 패턴:")
print("=" * 50)
print("""
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

mcp_config = {
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    }
}

async with MultiServerMCPClient(mcp_config) as client:
    # 1. MCP 서버에서 도구 가져오기
    mcp_tools = client.get_tools()

    # 2. 에이전트에 도구 전달
    agent = create_agent(
        model="gpt-4.1",
        tools=mcp_tools,
    )

    # 3. 에이전트 실행
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "2 + 3은?"}]}
    )
""")
print("핵심: client.get_tools()가 MCP 도구를 LangChain Tool로 자동 변환합니다.")

MCP 도구 → 에이전트 통합 패턴:

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

mcp_config = {
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    }
}

async with MultiServerMCPClient(mcp_config) as client:
    # 1. MCP 서버에서 도구 가져오기
    mcp_tools = client.get_tools()

    # 2. 에이전트에 도구 전달
    agent = create_agent(
        model="gpt-4.1",
        tools=mcp_tools,
    )

    # 3. 에이전트 실행
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "2 + 3은?"}]}
    )

핵심: client.get_tools()가 MCP 도구를 LangChain Tool로 자동 변환합니다.


## 11.7 다중 MCP 서버 연결

`MultiServerMCPClient`는 이름 그대로 여러 MCP 서버를 동시에 관리할 수 있습니다.

In [7]:
# 다중 MCP 서버 설정 예시
multi_server_config = {
    "math_server": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    },
    "weather_server": {
        "transport": "streamable_http",
        "url": "https://weather-mcp.example.com/mcp",
    },
    "database_server": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-postgres"],
        "env": {"DATABASE_URL": "postgresql://..."},
    },
}

print("다중 MCP 서버 설정:")
import json
print(json.dumps(multi_server_config, indent=2, ensure_ascii=False))
print()
print("사용 패턴:")
print("  async with MultiServerMCPClient(multi_server_config) as client:")
print("      tools = client.get_tools()  # 모든 서버의 도구를 한 번에 로드")
print("      # tools 리스트에 math, weather, database 도구가 모두 포함")
print()
print("참고: 기본적으로 stateless — 각 도구 호출마다 새 세션 생성 후 정리")

다중 MCP 서버 설정:
{
  "math_server": {
    "transport": "stdio",
    "command": "python",
    "args": [
      "math_server.py"
    ]
  },
  "weather_server": {
    "transport": "streamable_http",
    "url": "https://weather-mcp.example.com/mcp"
  },
  "database_server": {
    "transport": "stdio",
    "command": "npx",
    "args": [
      "-y",
      "@modelcontextprotocol/server-postgres"
    ],
    "env": {
      "DATABASE_URL": "postgresql://..."
    }
  }
}

사용 패턴:
  async with MultiServerMCPClient(multi_server_config) as client:
      tools = client.get_tools()  # 모든 서버의 도구를 한 번에 로드
      # tools 리스트에 math, weather, database 도구가 모두 포함

참고: 기본적으로 stateless — 각 도구 호출마다 새 세션 생성 후 정리


## 11.8 도구 인터셉터

**Tool Interceptor**는 MCP 도구 호출을 가로채는 미들웨어입니다. 런타임 컨텍스트에 접근하거나, 요청/응답을 수정하거나, 재시도 로직을 구현할 수 있습니다.

### 인터셉터 사용 사례

| 사용 사례 | 설명 |
|----------|------|
| 인증 주입 | 런타임에 사용자별 토큰 전달 |
| 요청 수정 | 도구 호출 파라미터 변환 |
| 응답 필터링 | 민감한 정보 제거 |
| 재시도 로직 | 실패 시 자동 재시도 |
| 로깅 | 도구 호출 추적 |

In [8]:
# 도구 인터셉터 예시 (개념 코드)
print("도구 인터셉터 패턴:")
print("=" * 50)
print("""
from langchain_mcp_adapters.client import MultiServerMCPClient

# 인터셉터 함수 정의
async def auth_interceptor(request, context):
    \"\"\"런타임 컨텍스트에서 사용자 토큰을 주입합니다.\"\"\" 
    user_token = context.get("user_token", "")
    request.params["auth_token"] = user_token
    return request

async def logging_interceptor(request, context):
    \"\"\"도구 호출을 로깅합니다.\"\"\"
    print(f"Tool call: {request.tool_name}")
    return request

# 인터셉터를 클라이언트에 전달
async with MultiServerMCPClient(
    config,
    interceptors=[auth_interceptor, logging_interceptor],
) as client:
    tools = client.get_tools()
""")
print("인터셉터는 도구 호출 전에 순서대로 실행됩니다.")

도구 인터셉터 패턴:

from langchain_mcp_adapters.client import MultiServerMCPClient

# 인터셉터 함수 정의
async def auth_interceptor(request, context):
    """런타임 컨텍스트에서 사용자 토큰을 주입합니다.""" 
    user_token = context.get("user_token", "")
    request.params["auth_token"] = user_token
    return request

async def logging_interceptor(request, context):
    """도구 호출을 로깅합니다."""
    print(f"Tool call: {request.tool_name}")
    return request

# 인터셉터를 클라이언트에 전달
async with MultiServerMCPClient(
    config,
    interceptors=[auth_interceptor, logging_interceptor],
) as client:
    tools = client.get_tools()

인터셉터는 도구 호출 전에 순서대로 실행됩니다.


## 11.9 커스텀 MCP 서버 작성

**FastMCP** 라이브러리를 사용하면 데코레이터로 간편하게 MCP 서버를 구축할 수 있습니다.

In [9]:
# 커스텀 MCP 서버 예시 (FastMCP)
print("커스텀 MCP 서버 작성 (FastMCP):")
print("=" * 50)
print("""
# my_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-tools")

@mcp.tool()
def add(a: int, b: int) -> int:
    \"\"\"두 수를 더합니다.\"\"\"
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    \"\"\"두 수를 곱합니다.\"\"\"
    return a * b

@mcp.resource("config://app")
def get_config() -> str:
    \"\"\"앱 설정을 반환합니다.\"\"\"
    return '{"version": "1.0", "debug": false}'

if __name__ == "__main__":
    mcp.run(transport="stdio")
""")
print("실행 방법:")
print("  1. Stdio: python my_server.py")
print("  2. HTTP:  mcp.run(transport='streamable-http', port=8080)")
print()
print("LangChain 연결:")
print('  config = {"my_tools": {"transport": "stdio", "command": "python", "args": ["my_server.py"]}}')

커스텀 MCP 서버 작성 (FastMCP):

# my_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-tools")

@mcp.tool()
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다."""
    return a * b

@mcp.resource("config://app")
def get_config() -> str:
    """앱 설정을 반환합니다."""
    return '{"version": "1.0", "debug": false}'

if __name__ == "__main__":
    mcp.run(transport="stdio")

실행 방법:
  1. Stdio: python my_server.py
  2. HTTP:  mcp.run(transport='streamable-http', port=8080)

LangChain 연결:
  config = {"my_tools": {"transport": "stdio", "command": "python", "args": ["my_server.py"]}}


## 11.10 요약

이 노트북에서 배운 내용:

| 주제 | 핵심 내용 |
|------|----------|
| **MCP 개념** | 외부 도구와 컨텍스트를 표준화된 방식으로 LLM에 제공하는 오픈 프로토콜입니다 |
| **Stdio 전송** | 로컬 서브프로세스를 통한 통신으로, 개발/테스트에 적합합니다 |
| **SSE/HTTP 전송** | 웹 기반 통신으로, 원격 서버 연결 및 인증을 지원합니다 |
| **에이전트 통합** | `client.get_tools()` → `create_agent(tools=mcp_tools)`로 연결합니다 |
| **다중 서버** | `MultiServerMCPClient`에 여러 서버를 설정하여 동시 관리합니다 |
| **인터셉터** | 도구 호출을 가로채 인증, 로깅, 수정 등의 미들웨어를 적용합니다 |
| **커스텀 서버** | `FastMCP`의 데코레이터로 간편하게 MCP 서버를 구축합니다 |

### 다음 단계
→ **[12. 프론트엔드 스트리밍](12_frontend_streaming.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [MCP (Model Context Protocol)](../docs/langchain/16-mcp.md)